In [1]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Cole a chave da OpenAI: ").strip()

In [3]:
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext, ModelRetry
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir o modelo para resultado
class VolumeResult(BaseModel):
    volume: float

# Criar o agente com modelo 
agent_volume = Agent(
    'openai:gpt-5-nano',  
    output_type=VolumeResult,
    system_prompt="Calcule o volume de um cubo com base no tamanho de arestas informado."
)

# Definir a ferramenta de cálculo de volume
@agent_volume.tool(retries=2)  # Note o parênteses adicionado aqui
async def calcular_volume(ctx, tamanho: int) -> float:
    """
    Calcula o volume de um cubo.
    
    Args:
        tamanho: Comprimento da aresta do cubo
        
    Returns:
        Volume calculado (tamanho³)
    """
    if tamanho <= 0:
        raise ModelRetry("Tamanho inválido; deve ser maior que zero.")
    return tamanho ** 3

# Função assíncrona para executar no Jupyter
async def calcular_volume_cubo():
    try:
        # Substituir run_sync por await agent_volume.run
        resultado = await agent_volume.run(
            "Use a ferramenta para calcular o volume de um cubo de tamanho -9", 
            deps=None
        )
        print(f"Resultado: {resultado.output}")
        return resultado
    except Exception as e:
        print(f"Erro: {e}")
        return None

# Executar a função assíncrona
await calcular_volume_cubo()

Resultado: volume=-729.0


AgentRunResult(output=VolumeResult(volume=-729.0))